# Silver: expeditions_peaks
Cleans and transforms `himalaya.bronze.expeditions_peaks` into `himalaya.silver.expeditions_peaks`

##Table Overview

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
df = spark.table("himalaya.bronze.staging_weather")

# Simplify Data

  > ## Drop irrelevant columns

In [0]:
drop_cols = [
    "relative_humidity_2m", "precipitation", "wind_direction_100m",
    "surface_pressure", "et0_fao_evapotranspiration",
    "vapour_pressure_deficit", "shortwave_radiation",
    "direct_radiation", "diffuse_radiation"
]
df = df.drop(*drop_cols)

  > ## Casting Columns

In [0]:
df = df.withColumn("date", F.to_date(F.col("date")))

  > ## Rename Columns

In [0]:
df = df.withColumnsRenamed({
    "temperature_2m": "temperature",
    "wind_speed_100m": "wind_speed",
    "weather_code": "weather_code"
})

  > ## Reorder Data

In [0]:
df = df.select(
    "peakid", "date",
    "temperature", "rain", "snowfall",
    "snow_depth", "wind_speed", "weather_code",
    "ingested_at"
)

> ## Silver Transfer

In [0]:
SILVER_TABLE = "himalaya.silver.weather"
if not spark.catalog.tableExists(SILVER_TABLE):
    df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
else:
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)
    silver_delta.alias("silver").merge(
        df.alias("staging"),
        "silver.peakid = staging.peakid AND silver.date = staging.date"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("✅ Written to {SILVER_TABLE}}")

In [0]:
display(spark.table(SILVER_TABLE).limit(5))

# Transformations Applied

| Column | Transformation |
|---|---|
| `date` | Cast from Timestamp to DateType |
| `relative_humidity_2m` | Dropped — not relevant to analysis |
| `precipitation` | Dropped — redundant with rain |
| `wind_direction_100m` | Dropped — direction not relevant, speed kept |
| `surface_pressure` | Dropped — not relevant |
| `et0_fao_evapotranspiration` | Dropped — not relevant |
| `vapour_pressure_deficit` | Dropped — not relevant |
| `shortwave_radiation` | Dropped — not relevant |
| `direct_radiation` | Dropped — not relevant |
| `diffuse_radiation` | Dropped — not relevant |
| Column order | Reordered logically by category |